# 04. 파생 피처 CatBoost

실행 결과는 `results/`에 저장됩니다.

In [ ]:
%pip install -q catboost==1.2.10

In [ ]:
from pathlib import Path
import sys

experiment_dir = Path.cwd() / "0826" if (Path.cwd() / "0826").exists() else Path.cwd()
if str(experiment_dir) not in sys.path:
    sys.path.insert(0, str(experiment_dir))


In [ ]:
"""상황·추세·표본 신뢰도 파생 피처를 추가한 CatBoost를 평가한다."""

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

from common import (
    CATBOOST_CAT_COLS,
    RESULTS_DIR,
    TARGET_COL,
    VALID_YEARS,
    Timer,
    brier_metrics,
    load_train,
    prepare_catboost_frame,
    print_metrics,
    save_json,
)


PARAMS = {
    "iterations": 700,
    "depth": 8,
    "learning_rate": 0.06,
    "loss_function": "Logloss",
    "eval_metric": "BrierScore",
    "l2_leaf_reg": 7.0,
    "random_strength": 0.5,
    "random_seed": 42,
    "thread_count": -1,
    "verbose": 100,
    "allow_writing_files": False,
}


def build_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    x = df.drop(columns=["row_id", TARGET_COL]).copy()
    x["count_state"] = (
        x["balls_before"].astype("Int64").astype("string")
        + "-"
        + x["strikes_before"].astype("Int64").astype("string")
    )
    x["hand_matchup"] = (
        x["pitcher_hand"].astype("string")
        + "-"
        + x["batter_hand"].astype("string")
    )
    x["inning_phase"] = pd.cut(
        x["inning"], bins=[-np.inf, 3, 6, np.inf], labels=["early", "middle", "late"]
    ).astype("string")
    x["is_scoring_position"] = (
        (x["runner_on_2b"] == 1) | (x["runner_on_3b"] == 1)
    ).astype("int8")
    x["is_bases_loaded"] = (x["base_state"] == "123").astype("int8")
    x["score_state"] = np.select(
        [x["score_diff_pitcher_team"] > 0, x["score_diff_pitcher_team"] < 0],
        ["leading", "trailing"],
        default="tied",
    )
    x["log_li"] = np.log1p(x["li"].clip(lower=0))

    for col in [c for c in x.columns if c.endswith("_n")]:
        x[f"log_{col}"] = np.log1p(x[col].clip(lower=0))
        x[f"missing_{col}"] = x[col].isna().astype("int8")

    trend_pairs = {
        "pitcher_success_trend_1v5": (
            "asof_pitcher_prev1_game_success_rate",
            "asof_pitcher_prev5_game_success_rate",
        ),
        "pitcher_success_trend_3v5": (
            "asof_pitcher_prev3_game_success_rate",
            "asof_pitcher_prev5_game_success_rate",
        ),
        "pitcher_middle_trend_1v5": (
            "asof_pitcher_prev1_game_middle_rate",
            "asof_pitcher_prev5_game_middle_rate",
        ),
        "pitcher_vs_batter_success_gap": (
            "asof_pitcher_success_rate",
            "asof_batter_success_rate",
        ),
    }
    for new_col, (left, right) in trend_pairs.items():
        x[new_col] = x[left] - x[right]

    extra_cat_cols = ["count_state", "hand_matchup", "inning_phase", "score_state"]
    cat_cols = [c for c in CATBOOST_CAT_COLS + extra_cat_cols if c in x]
    return prepare_catboost_frame(x, cat_cols), cat_cols


def main() -> None:
    train, _ = load_train()
    x, cat_cols = build_features(train)
    predictions, targets, years = [], [], []
    fold_results = []

    for valid_year in VALID_YEARS:
        train_mask = train["season"] < valid_year
        valid_mask = train["season"] == valid_year
        model = CatBoostClassifier(**PARAMS)
        with Timer() as timer:
            model.fit(
                x.loc[train_mask],
                train.loc[train_mask, TARGET_COL],
                cat_features=cat_cols,
                eval_set=(x.loc[valid_mask], train.loc[valid_mask, TARGET_COL]),
                early_stopping_rounds=100,
                use_best_model=True,
            )
            pred = model.predict_proba(x.loc[valid_mask])[:, 1]
        y = train.loc[valid_mask, TARGET_COL].to_numpy()
        metrics = brier_metrics(y, pred)
        metrics.update(
            {
                "valid_year": valid_year,
                "seconds": timer.seconds,
                "best_iteration": model.get_best_iteration(),
            }
        )
        fold_results.append(metrics)
        print_metrics(str(valid_year), metrics)
        predictions.append(pred.astype(np.float32))
        targets.append(y.astype(np.int8))
        years.append(np.full(len(y), valid_year, dtype=np.int16))

    all_pred = np.concatenate(predictions)
    all_y = np.concatenate(targets)
    all_year = np.concatenate(years)
    overall = brier_metrics(all_y, all_pred)
    print_metrics("OOF 전체", overall)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        RESULTS_DIR / "04_features_catboost_oof.npz",
        y=all_y,
        prediction=all_pred,
        year=all_year,
    )
    save_json(
        RESULTS_DIR / "04_features_catboost_metrics.json",
        {
            "model": "CatBoost with engineered features",
            "params": PARAMS,
            "feature_count": len(x.columns),
            "folds": fold_results,
            "overall": overall,
        },
    )
    model.save_model(RESULTS_DIR / "04_features_catboost_last_fold.cbm")


if __name__ == "__main__":
    main()

